In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import re
import random
import networkx as nx
from tqdm import tqdm
from scipy.sparse import coo_matrix
import community as community_louvain
from collections import defaultdict
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


df1 = pd.read_parquet('/COS_bert_word_embed.parquet', engine='pyarrow').set_index('word')
df1.columns = df1.index

df2 = pd.read_parquet('/COS_bert_word_topic_sim.parquet', engine='pyarrow').set_index('word')
df2.columns = df2.index

df3 = pd.read_parquet('/COS_lda_word_topic_prob.parquet', engine='pyarrow').set_index('word')
df3.columns = df3.index


# 단어 순서 섞어주기
words_shuffled = df1.index.tolist()
random.shuffle(words_shuffled)
df1 = df1.loc[words_shuffled, words_shuffled]
df2 = df2.loc[words_shuffled, words_shuffled]
df3 = df3.loc[words_shuffled, words_shuffled]

# 0 제거
df1[df1 < 0] = 0
df2[df2 < 0] = 0
df3[df3 < 0] = 0

In [2]:
df1

word,인수,주민,당부,시작,오픈,부동산,조사,거리,소양,확인,...,필기,기대감,콘텐츠들,논란,디자이너,사건,생태,수용,검사,단위
word,,,,,,,,,,,,,,,,,,,,,
인수,1.000000,0.541406,0.507308,0.472583,0.473823,0.581587,0.534695,0.522558,0.540198,0.553875,...,0.508920,0.509450,0.529675,0.541946,0.320746,0.604996,0.467637,0.645286,0.585428,0.587589
주민,0.541406,1.000000,0.570825,0.526697,0.542195,0.555448,0.498983,0.688470,0.525100,0.499364,...,0.470233,0.457991,0.466474,0.544115,0.401015,0.588249,0.487998,0.634791,0.579447,0.572093
당부,0.507308,0.570825,1.000000,0.795518,0.781408,0.311606,0.614614,0.647435,0.808878,0.679231,...,0.759584,0.713520,0.608712,0.791186,0.401627,0.781552,0.559033,0.792817,0.915404,0.880810
시작,0.472583,0.526697,0.795518,1.000000,0.836717,0.354838,0.643685,0.642839,0.752404,0.650187,...,0.716414,0.628271,0.602129,0.685909,0.482552,0.743516,0.575202,0.796253,0.831817,0.824569
오픈,0.473823,0.542195,0.781408,0.836717,1.000000,0.419047,0.662757,0.680031,0.746594,0.713503,...,0.713203,0.648431,0.652714,0.662274,0.470910,0.715201,0.544438,0.818933,0.838245,0.813813
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
사건,0.604996,0.588249,0.781552,0.743516,0.715201,0.461017,0.756028,0.664084,0.769845,0.725260,...,0.752393,0.621071,0.661452,0.844978,0.404333,1.000000,0.574340,0.829690,0.851174,0.802924
생태,0.467637,0.487998,0.559033,0.575202,0.544438,0.412737,0.607777,0.471815,0.654214,0.507253,...,0.603708,0.470822,0.564290,0.547516,0.426452,0.574340,1.000000,0.633406,0.598780,0.610064
수용,0.645286,0.634791,0.792817,0.796253,0.818933,0.523545,0.771250,0.726361,0.823405,0.753617,...,0.774813,0.672431,0.730234,0.756806,0.535329,0.829690,0.633406,1.000000,0.898137,0.883254


In [3]:
df2

word,인수,주민,당부,시작,오픈,부동산,조사,거리,소양,확인,...,필기,기대감,콘텐츠들,논란,디자이너,사건,생태,수용,검사,단위
word,,,,,,,,,,,,,,,,,,,,,
인수,1.000000,0.940580,0.957046,0.955293,0.961912,0.965609,0.975411,0.951870,0.962734,0.979099,...,0.963539,0.974907,0.962671,0.962435,0.937669,0.961095,0.981178,0.973855,0.968326,0.965205
주민,0.940580,1.000000,0.985124,0.977232,0.978340,0.935714,0.961982,0.994006,0.977742,0.969484,...,0.978551,0.979593,0.962122,0.983974,0.931679,0.986431,0.970955,0.975798,0.981640,0.980420
당부,0.957046,0.985124,1.000000,0.997099,0.995266,0.926233,0.972337,0.991695,0.995954,0.989588,...,0.994386,0.983516,0.981243,0.992468,0.950378,0.992491,0.983015,0.989691,0.997851,0.996913
시작,0.955293,0.977232,0.997099,1.000000,0.997907,0.926024,0.976964,0.989579,0.996724,0.989364,...,0.990059,0.981048,0.988608,0.990510,0.965102,0.991295,0.985689,0.993259,0.997967,0.998679
오픈,0.961912,0.978340,0.995266,0.997907,1.000000,0.935519,0.980584,0.991681,0.997310,0.993655,...,0.988616,0.980527,0.991763,0.989423,0.970891,0.990696,0.987031,0.996994,0.998280,0.998374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
사건,0.961095,0.986431,0.992491,0.991295,0.990696,0.954146,0.986886,0.994290,0.988539,0.982201,...,0.990757,0.992518,0.981910,0.998409,0.957405,1.000000,0.990840,0.989367,0.993190,0.992758
생태,0.981178,0.970955,0.983015,0.985689,0.987031,0.962835,0.994348,0.981254,0.985588,0.986883,...,0.979891,0.991253,0.989729,0.990807,0.969002,0.990840,1.000000,0.993503,0.989749,0.990670
수용,0.973855,0.975798,0.989691,0.993259,0.996994,0.949901,0.988603,0.987828,0.994051,0.993897,...,0.983073,0.982566,0.995352,0.988914,0.979894,0.989367,0.993503,1.000000,0.995636,0.996851


In [4]:
df3

word,인수,주민,당부,시작,오픈,부동산,조사,거리,소양,확인,...,필기,기대감,콘텐츠들,논란,디자이너,사건,생태,수용,검사,단위
word,,,,,,,,,,,,,,,,,,,,,
인수,1.000000,0.032004,0.010651,0.415731,0.446905,0.021511,0.160405,0.686984,0.007641,0.311857,...,0.256135,0.404895,0.172377,0.216009,0.233732,0.227239,0.154294,0.204037,0.312030,0.074286
주민,0.032004,1.000000,0.429965,0.459525,0.033279,0.134845,0.258561,0.026889,0.006785,0.506301,...,0.140652,0.040978,0.016656,0.658410,0.023224,0.050095,0.030270,0.331305,0.173265,0.430703
당부,0.010651,0.429965,1.000000,0.363826,0.035222,0.305466,0.041287,0.014444,0.004898,0.037360,...,0.021665,0.013211,0.013046,0.067122,0.019815,0.092059,0.030155,0.156785,0.385036,0.413728
시작,0.415731,0.459525,0.363826,1.000000,0.420835,0.238163,0.371908,0.557323,0.021550,0.501826,...,0.268455,0.465251,0.311324,0.490934,0.466255,0.360742,0.348420,0.467956,0.396418,0.492052
오픈,0.446905,0.033279,0.035222,0.420835,1.000000,0.080906,0.107215,0.310356,0.055925,0.247340,...,0.661838,0.408560,0.135487,0.108862,0.119268,0.161286,0.113130,0.263062,0.148711,0.188158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
사건,0.227239,0.050095,0.092059,0.360742,0.161286,0.055618,0.245749,0.388288,0.007674,0.163175,...,0.148395,0.384505,0.138755,0.375089,0.140825,1.000000,0.119602,0.499868,0.270319,0.391314
생태,0.154294,0.030270,0.030155,0.348420,0.113130,0.066485,0.228348,0.321029,0.064924,0.271808,...,0.039032,0.204655,0.616105,0.111887,0.138559,0.119602,1.000000,0.253673,0.433674,0.191318
수용,0.204037,0.331305,0.156785,0.467956,0.263062,0.067761,0.269849,0.271860,0.008421,0.431997,...,0.223767,0.253178,0.034440,0.384388,0.427454,0.499868,0.253673,1.000000,0.395348,0.606826


# Make Graph!

## individual graphs

# threshold 없음


In [5]:
# 함수: 유사도 행렬 → 단일 그래프 생성
def create_graph_from_df(df, label):
    G = nx.Graph()
    words = df.index.tolist()
    G.add_nodes_from(words)
    
    for i, word_i in tqdm(enumerate(df.index), total=len(df), desc=f"Creating {label}"):
        for j in range(i + 1, len(df.columns)):
            word_j = df.columns[j]
            weight = df.iat[i, j]
            G.add_edge(word_i, word_j, weight=weight, type=label)
    
    return G

# 개별 그래프 생성
G1 = create_graph_from_df(df1, 'sim1')
G2 = create_graph_from_df(df2, 'sim2')
G3 = create_graph_from_df(df3, 'sim3')

# 결과 출력
print(f"G1 (sim1) - 노드 수: {G1.number_of_nodes()}, 엣지 수: {G1.number_of_edges()}")
print(f"G2 (sim2) - 노드 수: {G2.number_of_nodes()}, 엣지 수: {G2.number_of_edges()}")
print(f"G3 (sim3) - 노드 수: {G3.number_of_nodes()}, 엣지 수: {G3.number_of_edges()}")


Creating sim3: 100%|██████████| 2597/2597 [00:33<00:00, 78.29it/s] 

G1 (sim1) - 노드 수: 2597, 엣지 수: 3370906
G2 (sim2) - 노드 수: 2597, 엣지 수: 3370906
G3 (sim3) - 노드 수: 2597, 엣지 수: 3370906


In [6]:
# 모든 weight 값 추출
weights = [data['weight'] for _, _, data in G1.edges(data=True)]

# 통계값 계산
print(f"엣지 개수: {len(weights)}")
print(f"최소 weight: {np.min(weights)}")
print(f"최대 weight: {np.max(weights)}")
print(f"평균 weight: {np.mean(weights):.2f}")
print(f"중앙값 weight: {np.median(weights)}")
print(f"표준편차: {np.std(weights):.2f}")


엣지 개수: 3370906
최소 weight: 0.0
최대 weight: 0.9958657439066876
평균 weight: 0.60
중앙값 weight: 0.6054310913910537
표준편차: 0.17


In [7]:
# 모든 weight 값 추출
weights = [data['weight'] for _, _, data in G2.edges(data=True)]

# 통계값 계산
print(f"엣지 개수: {len(weights)}")
print(f"최소 weight: {np.min(weights)}")
print(f"최대 weight: {np.max(weights)}")
print(f"평균 weight: {np.mean(weights):.2f}")
print(f"중앙값 weight: {np.median(weights)}")
print(f"표준편차: {np.std(weights):.2f}")


엣지 개수: 3370906
최소 weight: 0.0883820220823126
최대 weight: 0.9999488179664748
평균 weight: 0.98
중앙값 weight: 0.984784873535886
표준편차: 0.03


In [8]:
# 모든 weight 값 추출
weights = [data['weight'] for _, _, data in G3.edges(data=True)]

# 통계값 계산
print(f"엣지 개수: {len(weights)}")
print(f"최소 weight: {np.min(weights)}")
print(f"최대 weight: {np.max(weights)}")
print(f"평균 weight: {np.mean(weights):.2f}")
print(f"중앙값 weight: {np.median(weights)}")
print(f"표준편차: {np.std(weights):.2f}")


엣지 개수: 3370906
최소 weight: 0.0005424193774261279
최대 weight: 0.999999508369228
평균 weight: 0.19
중앙값 weight: 0.1265978945886176
표준편차: 0.18


## 각 그래프에 대한 community detection

In [9]:
def get_louvain_partition(G, label, res):
    partition = community_louvain.best_partition(G, weight='weight', resolution=res, random_state=RANDOM_STATE)
    return pd.DataFrame(list(partition.items()), columns=['word', f'community_{label}'])

df_comm1 = get_louvain_partition(G1, 'sim1', 1.2) # 1.2
df_comm2 = get_louvain_partition(G2, 'sim2', 1.1) # 1.01
df_comm3 = get_louvain_partition(G3, 'sim3', 3.3) # 3.3 

print(f"sim1 커뮤니티 수: {df_comm1['community_sim1'].nunique()}")
print(f"sim2 커뮤니티 수: {df_comm2['community_sim2'].nunique()}")
print(f"sim3 커뮤니티 수: {df_comm3['community_sim3'].nunique()}")

sim1 커뮤니티 수: 821
sim2 커뮤니티 수: 2550
sim3 커뮤니티 수: 626


In [10]:
# 단어 기준 병합
df_merged = df_comm1.merge(df_comm2, on='word').merge(df_comm3, on='word')
df_merged = df_merged.set_index('word')


In [11]:
df_merged

,community_sim1,community_sim2,community_sim3
word,,,
인수,0,0,58
주민,1,1,290
당부,2,2,509
시작,3,3,3
오픈,4,4,4
...,...,...,...
사건,498,2546,592
생태,156,2547,518
수용,789,2548,480


In [12]:
import gc

del df_comm1, df_comm2, df_comm3, G1, G2, G3
gc.collect()


23052

# Vectorize

## OneHotEncoder (Best)

In [13]:
encoder = OneHotEncoder(sparse_output=False)  
community_vectors = encoder.fit_transform(df_merged)
pd.DataFrame(community_vectors, index=df_merged.index)

,0,1,2,3,4,5,6,7,8,9,...,3987,3988,3989,3990,3991,3992,3993,3994,3995,3996
word,,,,,,,,,,,,,,,,,,,,,
인수,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
주민,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
당부,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
시작,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
오픈,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
사건,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
생태,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
수용,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
encoder = OneHotEncoder(sparse_output=False)  
community_vectors = encoder.fit_transform(df_merged)
community_vectors = pd.DataFrame(community_vectors, index=df_merged.index)

sim_matrix = cosine_similarity(community_vectors.values)

G_cluster = nx.Graph()
words = community_vectors.index.tolist()
G_cluster.add_nodes_from(words)

threshold = 0.3 
for i in tqdm(range(len(words)), desc="Building new graph"):
    for j in range(i + 1, len(words)):
        sim = sim_matrix[i, j]
        if sim >= threshold:
            G_cluster.add_edge(words[i], words[j], weight=sim)


print(f"G_cluster - 노드 수: {G_cluster.number_of_nodes()}, 엣지 수: {G_cluster.number_of_edges()}")

Building new graph: 100%|██████████| 2597/2597 [00:00<00:00, 3517.07it/s]

G_cluster - 노드 수: 2597, 엣지 수: 122992


In [15]:
# 모든 weight 값 추출
weights = [data['weight'] for _, _, data in G_cluster.edges(data=True)]

# 통계값 계산
print(f"엣지 개수: {len(weights)}")
print(f"최소 weight: {np.min(weights)}")
print(f"최대 weight: {np.max(weights)}")
print(f"평균 weight: {np.mean(weights):.2f}")
print(f"중앙값 weight: {np.median(weights)}")
print(f"표준편차: {np.std(weights):.2f}")


엣지 개수: 122992
최소 weight: 0.3333333333333334
최대 weight: 1.0000000000000002
평균 weight: 0.34
중앙값 weight: 0.3333333333333334
표준편차: 0.04


In [16]:
pd.DataFrame(weights)[0].value_counts()

0
0.333333    121228
0.666667      1757
1.000000         7
Name: count, dtype: int64

# Community Detection

In [17]:
res_range = np.arange(3.0, 3.1, 0.1)

resolutions = [round(x, 1) for x in res_range]

resolution_results = {}  # 결과 저장용 딕셔너리

for res in tqdm(resolutions, desc="Louvain by resolution"):
    print(f"\n================ Resolution: {res:.1f} ================")

    partition = community_louvain.best_partition(G_cluster, weight='weight', resolution=res, random_state=RANDOM_STATE)
    num_communities = len(set(partition.values()))

    community_groups = defaultdict(list)
    for node, comm_id in partition.items():
        community_groups[comm_id].append(node)

    community_groups = dict(sorted(community_groups.items(), key=lambda x: len(x[1]), reverse=True))

    # 결과 저장
    resolution_results[res] = {
        'num_communities': num_communities,
        'communities': community_groups
    }

    print(f"총 커뮤니티 수: {num_communities}")


Louvain by resolution:   0%|          | 0/2 [00:00<?, ?it/s]


================ Resolution: 3.0 ================


Louvain by resolution:  50%|█████     | 1/2 [00:00<00:00,  1.19it/s]

총 커뮤니티 수: 168

================ Resolution: 3.1 ================


Louvain by resolution: 100%|██████████| 2/2 [00:01<00:00,  1.19it/s]

총 커뮤니티 수: 168


In [18]:
def print_communities_by_resolution(res):
    print(f"\n Resolution: {res}")
    result = resolution_results.get(res)
    
    if result is None:
        print("해당 resolution 값의 결과가 없습니다.")
        return

    print(f"총 커뮤니티 수: {result['num_communities']}")
    
    for comm_id, words in result['communities'].items():
        print(f"\nCommunity {comm_id} ({len(words)}개 단어)")
        print(', '.join(words))


In [19]:
print_communities_by_resolution(3.0)


 Resolution: 3.0
총 커뮤니티 수: 168

Community 0 (134개 단어)
인수, 부동산, 대기업, 가격, 얼마, 저작권, 혁신적, 몸값, 영업, 홍보, 상용, 수집, 경제적, 기업가치, 자금, 투자금, 계약서, 크레딧, 게재, 사업자, 자본, 저장, 예약, 제품, 결제, 제조, 자원, 점유율, 창업, 산업현장, 사용률, 부문장, 복지, 공장, 기업가, 지급, 효율성, 식당, 가치, 효율, 고객, 수요, 보험, 자본주의, 마케팅, 금융, 보수, 창업자, 제조업, 품질, 성장, 금액, 성장세, 고객사들, 매출, 혁신, 유통, 경영학, 지속가능, 영업이익, 벤처, 수익률, 계산, 지역균형발전, 효율화, 자회사, 주식, 산업혁명, 투자, 카드, 계정, 매출액, 업그레이드, 기업들, 혁신상, 쇼핑, 비용, 거래, 경제성, 사업화, 상점, 요금, 수익화, 수익, 용량, 상품, 보급, 특허, 재정, 지원사업, 개척, 매장, 효율적, 예산, 수익성, 농업, 스토어, 비즈니스, 계약, 가치관, 맛집, 기업간거래, 공급, 업계, 수출, 임금, 중소기업, 은행, 판매자, 이코노미, 서점, 절약, 재산, 생산, 산업, 투자유치, 사업, 고객들, 상거래, 마켓, 생산성, 판매, 산업계, 냉장고, 교환, 회사들, 연평균, 구매, 소비자, 사용량, 사업부, 배송, 소비, 투자자

Community 5 (124개 단어)
조사, 인류, 우리말, 재난, 부모, 응답, 최종, 이동, 디바이드, 마지막, 태도, 월드, 약점, 인간, 오늘날, 외교, 문명, 탄소, 사장, 퍼포먼스, 주권, 중소, 수도, 진입, 근로자, 가족들, 지수, 국회, 독특, 전시장, 초래, 보고서, 묘사, 불평등, 협의회, 학위, 상승, 시사, 하위, 증가, 이래, 추가적, 소득, 영웅, 설명, 종료, 위기, 수입, 박사, 사무실, 분리, 자율주행, 과거, 제외, 세계관, 지향, 자동차, 숫자, 시민들, 민주주의, 노동, 직종, 시위, 컨텐츠, 마무리, 방한, 팬덤, 평등, 대책, 자산, 설명회, 

# Modularity

In [20]:
for i in res_range:
    partition = community_louvain.best_partition(G_cluster, weight='weight', resolution=round(i,2), random_state=RANDOM_STATE)
    modularity = community_louvain.modularity(partition, G_cluster, weight='weight')
    print(f"resolution:{round(i,2)}", "->", f"Modularity: {modularity:.4f}")

resolution:3.0 -> Modularity: 0.6101
resolution:3.1 -> Modularity: 0.6101


In [ ]:
import json

modularity_scores = {
    "modularity": community_louvain.modularity(community_louvain.best_partition(G_cluster, weight='weight', resolution=3.0, random_state=RANDOM_STATE), 
                                                   G_cluster, weight='weight'),
}

with open("/meta_community_bert_word_topic_change_.json", "w") as f:
    json.dump(modularity_scores, f, indent=4)

print("Modularity scores saved.")


# Coherence

In [ ]:
import pickle

# 불러오기
with open("/tokenized_docs.pkl", "rb") as f:
    tokenized_docs = pickle.load(f)

# Save Coherence

In [ ]:
import json
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# 원하는 resolution 값 지정
selected_resolution = 3.0

# resolution 결과에서 커뮤니티 단어 리스트 추출
communities = resolution_results[selected_resolution]['communities']

# 10개 미만 커뮤니티 제외
filtered_communities = [words for words in communities.values() if len(words) >= 10]

# Dictionary 및 Corpus 생성
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

# 다양한 coherence metric 계산
coherence_scores = {}
metrics = ['c_v', 'u_mass', 'c_uci', 'c_npmi']
for metric in metrics:
    cm = CoherenceModel(
        topics=filtered_communities,
        texts=tokenized_docs,
        corpus=corpus if metric in ['u_mass'] else None,
        dictionary=dictionary,
        coherence=metric
    )
    coherence_scores[metric] = cm.get_coherence()


try:
    
    w2v_model = Word2Vec(tokenized_docs, vector_size=100, window=5, min_count=2, workers=4, epochs=10,  seed=RANDOM_STATE)

    cm_w2v = CoherenceModel(
        topics=filtered_communities,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_w2v',
        keyed_vectors=w2v_model.wv
    )
    coherence_scores['c_w2v'] = cm_w2v.get_coherence().astype('float64')

except Exception as e:
    print("Word2Vec coherence 계산 실패:", e)
    coherence_scores['c_w2v'] = None


output_path = f"/meta_community_bert_word_topic_change_1.json"
with open(output_path, "w") as f:
    json.dump(coherence_scores, f, indent=4)

print(f"Coherence scores saved to {output_path}")

coherence_scores

